In [21]:
from utils.analysis.valuation import (
    CompanyAnalyzer,
    ComparisonAnalyzer,
    SectorAnalyzer,
    AnalysisWeights,
    CompanyReporter
)
from utils.tools.config import INVESTMENT_PROFILES

In [22]:
# 📊 CONFIGURACIÓN DEL ANÁLISIS
# Los pesos determinan la importancia de cada categoría en el score final

# ============================================================================
# SELECCIONA TU PERFIL DE INVERSIÓN
# ============================================================================
# Opciones: 'balanced', 'value', 'growth', 'quality'
# - balanced: Equilibrado en todas las métricas (default)
# - value: Énfasis en valoración (35%) - para value investors
# - growth: Énfasis en crecimiento (45%) - para growth investors
# - quality: Énfasis en rentabilidad y salud (35% cada uno)
# ============================================================================

PROFILE_NAME = 'quality'  

# Cargar perfil desde config
if PROFILE_NAME and PROFILE_NAME in INVESTMENT_PROFILES:
    profile = INVESTMENT_PROFILES[PROFILE_NAME]
    custom_weights = AnalysisWeights(
        profitability=profile['profitability'],
        financial_health=profile['financial_health'],
        growth=profile['growth'],
        efficiency=profile['efficiency'],
        valuation=profile['valuation']
    )
    print(f"✅ Perfil seleccionado: {PROFILE_NAME.upper()}")
    print(f"   {profile['description']}\n")
else:
    # Usar pesos por defecto (balanced)
    custom_weights = None
    print("✅ Usando pesos por defecto (Balanced)\n")

# Inicializar componentes
analyzer = CompanyAnalyzer(weights=custom_weights)
comparator = ComparisonAnalyzer(company_analyzer=analyzer)
sector_analyzer = SectorAnalyzer(company_analyzer=analyzer)
reporter = CompanyReporter()

# Mostrar configuración actual
print("📋 Pesos configurados:")
print(f"   Rentabilidad:      {analyzer.weights.profitability:.0%}")
print(f"   Salud Financiera:  {analyzer.weights.financial_health:.0%}")
print(f"   Crecimiento:       {analyzer.weights.growth:.0%}")
print(f"   Eficiencia:        {analyzer.weights.efficiency:.0%}")
print(f"   Valoración:        {analyzer.weights.valuation:.0%}")

✅ Perfil seleccionado: QUALITY
   Para inversores que priorizan calidad (alto ROE, bajo debt)

📋 Pesos configurados:
   Rentabilidad:      35%
   Salud Financiera:  35%
   Crecimiento:       15%
   Eficiencia:        10%
   Valoración:        5%


In [23]:
# 🔍 ANÁLISIS INDIVIDUAL DE EMPRESA
# Analiza fundamentalmente una empresa usando el perfil configurado

TICKER_TO_ANALYZE = "AAPL" 

print(f"🔍 Analizando {TICKER_TO_ANALYZE} con perfil {PROFILE_NAME.upper()}...\n")

result = analyzer.analyze(TICKER_TO_ANALYZE)

if result.get('success'):
    # Mostrar reporte completo
    print(reporter.render(result))
    
    # Información adicional
    print("\n" + "="*65)
    print("💡 INTERPRETACIÓN:")
    print("="*65)
    conclusion = result['conclusion']
    print(f"Calificación: {conclusion['overall']}")
    print(f"Score Total:  {result['scores']['total']:.1f}/100")
    
    # Mostrar alertas si existen
    all_alerts = []
    for category in ['profitability', 'financial_health', 'growth', 'efficiency', 'valuation']:
        alerts = result.get(category, {}).get('alerts', [])
        all_alerts.extend([(category, alert) for alert in alerts])
    
    if all_alerts:
        print(f"\n⚠️  {len(all_alerts)} ALERTAS DETECTADAS:")
        for cat, alert in all_alerts[:5]:  # Mostrar máximo 5
            print(f"   [{cat}] {alert}")
else:
    print(f"❌ Error: {result.get('error')}")

🔍 Analizando AAPL con perfil QUALITY...

ANÁLISIS: AAPL (AAPL)
Sector: Technology | Industria: Consumer Electronics
País: N/A | Moneda: USD

─────────────────────────────────────────────────────────────────
RESUMEN DE SCORES
─────────────────────────────────────────────────────────────────
  🟢 Rentabilidad       [████████████████████]  100.0
  🟠 Salud Financiera   [███████████░░░░░░░░░]   58.4
  🟡 Crecimiento        [██████████████░░░░░░]   73.2
  ❓ Eficiencia         [????????????????????]    N/A
  🔴 Valoración         [██░░░░░░░░░░░░░░░░░░]   10.8
─────────────────────────────────────────────────────────────────
  🟡 TOTAL              [██████████████░░░░░░]   74.4
  📋 Conclusión: {'overall': 'BUENA', 'score': np.float64(74.4154902500865)}

─────────────────────────────────────────────────────────────────
RENTABILIDAD (Score: 100.0)
─────────────────────────────────────────────────────────────────
  ROIC:             N/A          N/A
  ROE:              171.4%       excellent
  ROA:  

In [24]:
# ⚖️  COMPARACIÓN DE MÚLTIPLES EMPRESAS
# Compara fundamentales de varias empresas del mismo sector

TICKERS_TO_COMPARE = ["AAPL", "MSFT", "GOOGL", "META"] 

print(f"⚖️  Comparando {len(TICKERS_TO_COMPARE)} empresas...\n")

comparison = comparator.compare(TICKERS_TO_COMPARE)

if comparison.get('success'):
    # Mostrar tabla resumen
    print("📊 TABLA COMPARATIVA:\n")
    display(comparison['summary_df'])
    
    print(f"\n✅ {comparison['valid_count']}/{len(TICKERS_TO_COMPARE)} empresas analizadas exitosamente")
else:
    print(f"❌ Error en comparación: {comparison.get('error')}")

⚖️  Comparando 4 empresas...

📊 TABLA COMPARATIVA:



,Ticker,Nombre,Sector,Rentabilidad,Salud,Crecimiento,Eficiencia,Valoración,Total,Conclusión
0,AAPL,Apple Inc.,Technology,100.000000,58.420135,73.250000,NaN,10.787879,74.415490,BUENA
1,MSFT,Microsoft Corporation,Technology,98.248255,71.616439,58.687500,NaN,15.039669,76.675279,BUENA
2,GOOGL,Alphabet Inc.,Communication Services,100.000000,79.192754,70.729167,NaN,13.481670,82.223247,EXCELENTE
3,META,"Meta Platforms, Inc.",Communication Services,98.503493,77.683832,38.500000,NaN,23.596040,76.244851,BUENA



✅ 4/4 empresas analizadas exitosamente


In [25]:
# 🏆 RANKING DE EMPRESAS
# Ordena empresas por score total

if comparison.get('success'):
    print("\n🏆 RANKING POR SCORE TOTAL:\n")
    print(f"{'Rank':<6} {'Ticker':<8} {'Empresa':<32} {'Score':<8} {'Conclusión'}")
    print("-" * 70)
    
    for item in comparison['ranking']:
        # Emojis para top 3
        if item['rank'] == 1:
            emoji = "🥇"
        elif item['rank'] == 2:
            emoji = "🥈"
        elif item['rank'] == 3:
            emoji = "🥉"
        else:
            emoji = "  "
        
        name = item['name'][:30] if len(item['name']) > 30 else item['name']
        print(f"{emoji} #{item['rank']:<3} {item['ticker']:<8} {name:<32} "
              f"{item['score']:5.1f}  {item['conclusion']}")


🏆 RANKING POR SCORE TOTAL:

Rank   Ticker   Empresa                          Score    Conclusión
----------------------------------------------------------------------
🥇 #1   GOOGL    GOOGL                             82.2  {'overall': 'EXCELENTE', 'score': np.float64(82.22324697136591)}
🥈 #2   MSFT     MSFT                              76.7  {'overall': 'BUENA', 'score': np.float64(76.67527927976194)}
🥉 #3   META     META                              76.2  {'overall': 'BUENA', 'score': np.float64(76.24485095072558)}
   #4   AAPL     AAPL                              74.4  {'overall': 'BUENA', 'score': np.float64(74.4154902500865)}


In [26]:
# 📊 LÍDERES POR CATEGORÍA
# Identifica el mejor y peor en cada dimensión

if comparison.get('success'):
    category_names = {
        'profitability': 'Rentabilidad',
        'financial_health': 'Salud Financiera',
        'growth': 'Crecimiento',
        'efficiency': 'Eficiencia',
        'valuation': 'Valoración'
    }
    
    print("\n📊 LÍDERES POR CATEGORÍA:\n")
    
    for cat, data in comparison['category_leaders'].items():
        name = category_names.get(cat, cat)
        best = data['best']
        worst = data['worst']
        
        print(f"{name}:")
        print(f"  🟢 Mejor:  {best['ticker']:<6} - {best['name'][:25]:<25} (Score: {best['score']:5.1f})")
        print(f"  🔴 Peor:   {worst['ticker']:<6} - {worst['name'][:25]:<25} (Score: {worst['score']:5.1f})")
        print()


📊 LÍDERES POR CATEGORÍA:

Rentabilidad:
  🟢 Mejor:  AAPL   - AAPL                      (Score: 100.0)
  🔴 Peor:   MSFT   - MSFT                      (Score:  98.2)

Salud Financiera:
  🟢 Mejor:  GOOGL  - GOOGL                     (Score:  79.2)
  🔴 Peor:   AAPL   - AAPL                      (Score:  58.4)

Crecimiento:
  🟢 Mejor:  AAPL   - AAPL                      (Score:  73.2)
  🔴 Peor:   META   - META                      (Score:  38.5)

Valoración:
  🟢 Mejor:  META   - META                      (Score:  23.6)
  🔴 Peor:   AAPL   - AAPL                      (Score:  10.8)



In [27]:
# 📈 ESTADÍSTICAS DEL GRUPO
# Análisis estadístico de las empresas comparadas

if comparison.get('success'):
    stats = comparison['group_stats']
    
    print("\n📈 ESTADÍSTICAS DEL GRUPO:\n")
    
    categories = [
        ('Total', 'total'),
        ('Rentabilidad', 'profitability'),
        ('Salud Financiera', 'financial_health'),
        ('Crecimiento', 'growth'),
        ('Eficiencia', 'efficiency'),
        ('Valoración', 'valuation')
    ]
    
    print(f"{'Categoría':<20} {'Media':<8} {'Mediana':<10} {'Min':<8} {'Max':<8} {'Std':<8}")
    print("-" * 75)
    
    for name, key in categories:
        if key in stats:
            s = stats[key]
            print(f"{name:<20} {s['mean']:6.1f}   {s['median']:7.1f}    "
                  f"{s['min']:6.1f}   {s['max']:6.1f}   {s['std']:6.1f}")


📈 ESTADÍSTICAS DEL GRUPO:

Categoría            Media    Mediana    Min      Max      Std     
---------------------------------------------------------------------------
Total                  77.4      76.5      74.4     82.2      2.9
Rentabilidad           99.2      99.3      98.2    100.0      0.8
Salud Financiera       71.7      74.7      58.4     79.2      8.2
Crecimiento            60.3      64.7      38.5     73.2     13.7
Valoración             15.7      14.3      10.8     23.6      4.8


In [28]:
# 🏭 ANÁLISIS VS PEERS DEL SECTOR
# Compara una empresa específica con competidores del sector

TARGET_COMPANY = "AAPL"  # ← Empresa objetivo
PEERS = ["MSFT", "GOOGL", "META", "AMZN"]  # ← Competidores

print(f"🔬 Analizando {TARGET_COMPANY} vs peers del sector...\n")

sector_result = sector_analyzer.analyze_vs_peers(
    ticker=TARGET_COMPANY,
    peers=PEERS
)

if sector_result.get('success'):
    print(f"EMPRESA: {sector_result['company']['company_name']} ({TARGET_COMPANY})") 
    print(f"Sector: {sector_result['sector']}")
    print(f"Industria: {sector_result['industry']}")
    print(f"Peers analizados: {sector_result['peer_count']}\n")
    
    # Posición relativa
    rel_pos = sector_result['relative_position']
    category_names = {
        'profitability': 'Rentabilidad',
        'financial_health': 'Salud Financiera',
        'growth': 'Crecimiento',
        'efficiency': 'Eficiencia',
        'valuation': 'Valoración',
        'total': 'TOTAL'
    }
    
    print("POSICIÓN RELATIVA:\n")
    
    for cat, data in rel_pos.items():
        if isinstance(data, dict) and 'company_score' in data:
            name = category_names.get(cat, cat.upper())
            diff = data['vs_avg']
            arrow = "↑" if diff > 0 else "↓" if diff < 0 else "="
            
            # Color basado en diferencia
            if diff > 5:
                color = "🟢"
            elif diff < -5:
                color = "🔴"
            else:
                color = "🟡"
            
            print(f"{name}:")
            print(f"  Score empresa:   {data['company_score']:5.1f}")
            print(f"  Promedio peers:  {data['peer_avg']:5.1f}")
            print(f"  {color} Diferencia:    {diff:+5.1f} {arrow}")
            print(f"  Ranking:         #{data['rank']} de {data['total_compared']}")
            print()
    
    # Percentil
    if 'percentiles' in sector_result and 'percentile' in sector_result['percentiles']:
        pct = sector_result['percentiles']
        print(f"📊 PERCENTIL: {pct['percentile']:.1f}%")
        print(f"   Interpretación: {pct['interpretation']}")
        print(f"   Tamaño muestra: {pct['sample_size']} empresas")
else:
    print(f"❌ Error: {sector_result.get('error')}")

🔬 Analizando AAPL vs peers del sector...

EMPRESA: Apple Inc. (AAPL)
Sector: Technology
Industria: Consumer Electronics
Peers analizados: 4

POSICIÓN RELATIVA:

Rentabilidad:
  Score empresa:   100.0
  Promedio peers:   89.7
  🟢 Diferencia:    +10.3 ↑
  Ranking:         #4 de 5

Salud Financiera:
  Score empresa:    58.4
  Promedio peers:   73.3
  🔴 Diferencia:    -14.9 ↓
  Ranking:         #1 de 5

Crecimiento:
  Score empresa:    73.2
  Promedio peers:   59.3
  🟢 Diferencia:    +13.9 ↑
  Ranking:         #5 de 5

Valoración:
  Score empresa:    10.8
  Promedio peers:   19.0
  🔴 Diferencia:     -8.2 ↓
  Ranking:         #1 de 5

TOTAL:
  Score empresa:    74.4
  Promedio peers:   74.3
  🟡 Diferencia:     +0.1 ↑
  Ranking:         #2 de 5

📊 PERCENTIL: 20.0%
   Interpretación: Por debajo del promedio
   Tamaño muestra: 5 empresas


In [29]:
# 🎯 COMPARAR DIFERENTES PERFILES
# Analiza la misma empresa con diferentes estrategias de inversión

TEST_TICKER = "AAPL"  # ← Cambia aquí

print(f"🔬 Analizando {TEST_TICKER} con TODOS los perfiles disponibles:\n")
print(f"{'Perfil':<20} {'Score':<8} {'Conclusión':<15} {'Descripción'}")
print("=" * 80)

for profile_name, profile_config in INVESTMENT_PROFILES.items():
    # Crear analyzer con este perfil
    weights = AnalysisWeights(
        profitability=profile_config['profitability'],
        financial_health=profile_config['financial_health'],
        growth=profile_config['growth'],
        efficiency=profile_config['efficiency'],
        valuation=profile_config['valuation']
    )
    
    profile_analyzer = CompanyAnalyzer(weights=weights)
    result = profile_analyzer.analyze(TEST_TICKER)
    
    if result.get('success'):
        score = result['scores']['total']
        conclusion = result['conclusion']['overall']
        desc = profile_config['description']
        
        print(f"{profile_name.upper():<20} {score:5.1f}    {conclusion:<15} {desc}")
    else:
        print(f"{profile_name.upper():<20} ERROR")

print("\n💡 Observación: El mismo activo puede tener diferente evaluación según tu estrategia")

🔬 Analizando AAPL con TODOS los perfiles disponibles:

Perfil               Score    Conclusión      Descripción
BALANCED              66.8    BUENA           Para inversores que buscan equilibrio
VALUE                 50.8    REGULAR         Para inversores de valor (bajo P/E, alto dividend yield)
GROWTH                64.8    REGULAR         Para inversores de crecimiento (alto revenue growth)
QUALITY               74.4    BUENA           Para inversores que priorizan calidad (alto ROE, bajo debt)

💡 Observación: El mismo activo puede tener diferente evaluación según tu estrategia


In [30]:
# 🏭 ANÁLISIS DE SECTOR COMPLETO
# Analiza un grupo grande de empresas del mismo sector

SECTOR_NAME = "Technology"  # Para referencia
SECTOR_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "META", "NVDA",
    "AMZN", "TSLA", "CRM", "ADBE", "INTC"
]

print(f"🏭 ANÁLISIS DE SECTOR: {SECTOR_NAME}")
print(f"   Analizando {len(SECTOR_TICKERS)} empresas...\n")

sector_comparison = comparator.compare(SECTOR_TICKERS)

if sector_comparison.get('success'):
    ranking = sector_comparison['ranking']
    
    # Top 5
    print("🏆 TOP 5 DEL SECTOR:\n")
    for item in ranking[:5]:
        emoji = "🥇" if item['rank'] == 1 else "🥈" if item['rank'] == 2 else "🥉" if item['rank'] == 3 else "  "
        print(f"{emoji} {item['ticker']:<6} Score: {item['score']:5.1f} - {item['conclusion']}")
    
    # Bottom 5
    print(f"\n⚠️  BOTTOM 5 DEL SECTOR:\n")
    for item in ranking[-5:]:
        print(f"   {item['ticker']:<6} Score: {item['score']:5.1f} - {item['conclusion']}")
    
    # Estadísticas
    scores = [item['score'] for item in ranking]
    print(f"\n📊 ESTADÍSTICAS DEL SECTOR:")
    print(f"   Empresas analizadas: {len(scores)}")
    print(f"   Score promedio:      {sum(scores)/len(scores):.1f}")
    print(f"   Score mediano:       {sorted(scores)[len(scores)//2]:.1f}")
    print(f"   Mejor score:         {max(scores):.1f}")
    print(f"   Peor score:          {min(scores):.1f}")
    print(f"   Desviación estándar: {(sum((x - sum(scores)/len(scores))**2 for x in scores) / len(scores))**0.5:.1f}")
else:
    print(f"❌ Error: {sector_comparison.get('error')}")

🏭 ANÁLISIS DE SECTOR: Technology
   Analizando 10 empresas...

🏆 TOP 5 DEL SECTOR:

🥇 NVDA   Score:  83.6 - {'overall': 'EXCELENTE', 'score': np.float64(83.57162822044991)}
🥈 GOOGL  Score:  82.2 - {'overall': 'EXCELENTE', 'score': np.float64(82.22322496004797)}
🥉 MSFT   Score:  76.7 - {'overall': 'BUENA', 'score': np.float64(76.67527927976194)}
   META   Score:  76.2 - {'overall': 'BUENA', 'score': np.float64(76.24485095072558)}
   AAPL   Score:  74.4 - {'overall': 'BUENA', 'score': np.float64(74.4154902500865)}

⚠️  BOTTOM 5 DEL SECTOR:

   ADBE   Score:  73.9 - {'overall': 'BUENA', 'score': np.float64(73.91878392425429)}
   CRM    Score:  68.1 - {'overall': 'BUENA', 'score': np.float64(68.05587303371674)}
   AMZN   Score:  62.2 - {'overall': 'REGULAR', 'score': np.float64(62.227293209589625)}
   TSLA   Score:  53.6 - {'overall': 'REGULAR', 'score': np.float64(53.57602750734977)}
   INTC   Score:  34.7 - {'overall': 'DÉBIL', 'score': np.float64(34.718914586836895)}

📊 ESTADÍSTICAS DEL

In [31]:
# ℹ️ PERFILES DE INVERSIÓN DISPONIBLES
# Referencia rápida de todos los perfiles configurados

print("📋 PERFILES DE INVERSIÓN DISPONIBLES:\n")
print("=" * 80)

for i, (profile_name, config) in enumerate(INVESTMENT_PROFILES.items(), 1):
    print(f"\n{i}️⃣  {profile_name.upper()}")
    print(f"   {config['description']}")
    print(f"   Pesos:")
    print(f"      • Rentabilidad:     {config['profitability']:.0%}")
    print(f"      • Salud Financiera: {config['financial_health']:.0%}")
    print(f"      • Crecimiento:      {config['growth']:.0%}")
    print(f"      • Eficiencia:       {config['efficiency']:.0%}")
    print(f"      • Valoración:       {config['valuation']:.0%}")

print("\n" + "=" * 80)
print("💡 TIP: Cambia PROFILE_NAME en Cell 1 para seleccionar tu estrategia")
print("📚 Los perfiles están definidos en: utils/tools/config.py")

📋 PERFILES DE INVERSIÓN DISPONIBLES:


1️⃣  BALANCED
   Para inversores que buscan equilibrio
   Pesos:
      • Rentabilidad:     30%
      • Salud Financiera: 30%
      • Crecimiento:      15%
      • Eficiencia:       10%
      • Valoración:       15%

2️⃣  VALUE
   Para inversores de valor (bajo P/E, alto dividend yield)
   Pesos:
      • Rentabilidad:     20%
      • Salud Financiera: 25%
      • Crecimiento:      10%
      • Eficiencia:       10%
      • Valoración:       35%

3️⃣  GROWTH
   Para inversores de crecimiento (alto revenue growth)
   Pesos:
      • Rentabilidad:     15%
      • Salud Financiera: 15%
      • Crecimiento:      45%
      • Eficiencia:       10%
      • Valoración:       15%

4️⃣  QUALITY
   Para inversores que priorizan calidad (alto ROE, bajo debt)
   Pesos:
      • Rentabilidad:     35%
      • Salud Financiera: 35%
      • Crecimiento:      15%
      • Eficiencia:       10%
      • Valoración:       5%

💡 TIP: Cambia PROFILE_NAME en Cell 1 para selecc